# Projeto 3: Diagnóstico de Saúde Pública (Heart Disease)

## Objetivo
Prever a presença de doença cardíaca (`target = 1`) em pacientes a partir de exames e dados demográficos. A meta de negócio é obter um **Recall mínimo de 0.80 para a classe positiva (1)** no conjunto de teste.

## Por que Recall?
No contexto de saúde pública e medicina, o custo de um **Falso Negativo** (dizer que um paciente está saudável quando na verdade ele tem problemas cardíacos) é imensamente maior do que um **Falso Positivo** (dizer que ele tem problema cardíaco, forçando um exame confirmatório). Portanto, precisamos de uma sensibilidade (Recall) muito alta!

## Desafios
1. **Mapeamento de Tipos de Dados:** Várias colunas são codificadas numericamente no CSV, mas são conceitualmente categóricas (ex: `sex`, `cp` - tipo de dor no peito, `fbs` - açúcar no sangue, `restecg` - eletrocardiograma, `exang` - angina, `slope`, `ca` - número de vasos, `thal`). Você precisará tratá-las de forma correta.
2. **Ajuste de Threshold:** Às vezes, o threshold padrão de `0.5` na predição de probabilidade gera um Recall baixo. Você aprenderá a obter probabilidades via `predict_proba()` e aplicar um limiar menor (ex: `0.4` ou `0.3`) no conjunto de teste para otimizar a métrica de negócio.
3. **Deploy:** Salvar o pipeline final e simular o deploy.

## Passo 1: Importações e Carga dos Dados
Baixe os dados clínicos diretamente do link público e realize a divisão em treino (80%) e teste (20%).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# URL direta do dataset Heart Disease (UCI)
DATA_URL = "https://raw.githubusercontent.com/amankharwal/Website-data/main/heart.csv"

# 1. Carregar os dados
df = pd.read_csv(DATA_URL)
print(f"Formato dos dados: {df.shape}")
df.head()

In [ ]:
# TODO: Defina as features (X) e o target (y). O target é a coluna 'target'
X = None
y = None

# TODO: Divida os dados em treino (80%) e teste (20%) com random_state=42. Use stratify=y.
X_train, X_test, y_train, y_test = None, None, None, None

print(f"Treino: {X_train.shape if X_train is not None else 'Não implementado'}, Teste: {X_test.shape if X_test is not None else 'Não implementado'}")

## Passo 2: Construção do Pré-processador (`ColumnTransformer`)
Atenção: identifique as colunas de fato contínuas (ex: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`) e as categóricas (mesmo as que estão representadas por números, ex: `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`).
- Numéricas: Imputação + Normalização
- Categóricas: Imputação + OneHotEncoder

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: Defina as listas de colunas numéricas e categóricas com base no dataset
num_features = []
cat_features = []

# TODO: Monte as pipelines de transformação
num_transformer = None
cat_transformer = None

# TODO: Crie o ColumnTransformer
preprocessor = None

## Passo 3: Criação do Pipeline Final com o Estimador
Combine o pré-processador com um classificador (ex: `LogisticRegression`, `KNeighborsClassifier` ou `SVC`).

In [ ]:
from sklearn.linear_model import LogisticRegression # Exemplo

# TODO: Instancie o classificador
# TODO: Crie o pipeline completo
pipeline = None

# TODO: Treine (fit) o pipeline no conjunto de treino


## Passo 4: Avaliação Básica (Threshold Padrão = 0.5)
Calcule a Matriz de Confusão e o Recall obtido usando a chamada clássica `predict(X_test)`.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

# TODO: Faça a predição no conjunto de teste
y_pred = None

# TODO: Exiba a matriz de confusão e o classification report

# TODO: Calcule e exiba o Recall para a classe 1
recall_padrao = None
print(f"Recall (Threshold 0.5): {recall_padrao}")

## Passo 5: Otimização do Limiar de Decisão (Threshold Tuning)
Utilize as probabilidades de predição para alterar o threshold padrão. Nosso objetivo é capturar o máximo de doentes possíveis. Teste diminuir o threshold (ex: para 0.4 ou 0.3) e veja o impacto no Recall e na Precisão.

In [ ]:
# TODO: Obtenha as probabilidades de predição da classe positiva usando pipeline.predict_proba(X_test)[:, 1]
y_probs = None

# TODO: Teste um novo threshold menor, como 0.35 ou 0.40
threshold = 0.40
y_pred_novo = (y_probs >= threshold).astype(int) if y_probs is not None else None

# TODO: Imprima a nova Matriz de Confusão e o novo Recall da classe 1
new_recall = None
print(f"Novo Recall (Threshold {threshold}): {new_recall}")

# TODO: Verifique se a meta (Recall > 0.80) foi atingida

## Passo 6: Serialização (Deploy)
Salve o pipeline em disco.

In [ ]:
import joblib

# TODO: Salve o modelo como 'pipeline_saude.joblib'


## Passo 7: Simulação de Produção com Threshold Customizado
Construa a função de predição simulando a API de saúde pública. Ela deve:
1. Receber o payload contendo dados clínicos de um novo paciente.
2. Carregar o arquivo do pipeline.
3. Fazer predição de probabilidade.
4. Aplicar o threshold ajustado na Fase 5 para decidir se o paciente deve ou não ser encaminhado para exames urgentes (Classe 1).

In [ ]:
# Payload que vem da API de Produção (com valores brutos e strings)
payload_bruto = {
    "age": 58,
    "sex": 1,         # Categórica
    "cp": 2,          # Categórica (dor tipo 2)
    "trestbps": 140,
    "chol": 211,
    "fbs": 1,
    "restecg": 0,
    "thalach": 165,
    "exang": 0,
    "oldpeak": 0.0,
    "slope": 2,
    "ca": 0,
    "thal": 2
}

# TODO: Crie a função de predição aplicando o threshold customizado
def predizer_saude(payload, threshold=0.40):
    # 1. Carregar pipeline de 'pipeline_saude.joblib'
    # 2. Converter payload em DataFrame
    # 3. Chamar predict_proba para pegar a probabilidade da classe 1
    # 4. Retornar 1 se a probabilidade >= threshold, senão 0
    return 0

resultado = predizer_saude(payload_bruto, threshold=0.40)
print(f"Paciente com risco de doença cardíaca? {'SIM (Enviar para exames)' if resultado == 1 else 'NÃO'}")